# SetFit baseline reproduction — NLBSE'24Reproduces the organisers' published baseline on a GPU, because the CPU on thedevelopment machine needs two to four hours for the same work.Everything else in the project is measured against this number, so it has to bereproduced rather than quoted. The configuration is copied exactly from theorganisers' notebook: `all-mpnet-base-v2`, batch `(16, 2)`, one epoch, twentycontrastive iterations, seed 42, **one model per project**.Two published figures exist for that configuration and they disagree — thecompetition README says **0.8270**, the result file committed in the samerepository says **0.8240**. Landing between them is a successful reproduction.Chasing the third decimal place would be chasing noise: the measured detectionthreshold of this test set is 3.0 F1 points (`docs/benchmark-audit.md`).---### Before running**Runtime → Change runtime type → T4 GPU.** The next cell refuses to continuewithout one, because on CPU this notebook would run for hours rather thanminutes.Expected wall time on a T4: about 15 minutes.

## 1. Check the runtime

In [ ]:
import subprocess, systry:    import torchexcept ImportError:    sys.exit("torch is missing - this notebook expects a Colab runtime.")print("torch", torch.__version__)if not torch.cuda.is_available():    raise SystemExit(        "No GPU visible.\n"        "Runtime -> Change runtime type -> Hardware accelerator: T4 GPU, "        "then run this cell again.\n"        "Running the official preset on CPU takes two to four hours; that is "        "the whole reason this notebook exists."    )print("device:", torch.cuda.get_device_name(0))print(subprocess.run(["nvidia-smi", "--query-gpu=memory.total,memory.free",                      "--format=csv"], capture_output=True, text=True).stdout)

## 2. Get the project codeThe repository is **private**, so Colab cannot clone it anonymously. Pick one:**Option A — clone with a token** (fastest, and lets you pull updates later).Create a fine-grained personal access token with *Contents: read* on this onerepository at <https://github.com/settings/personal-access-tokens>, then paste itwhen prompted. It is read with `getpass`, so it never appears in the notebookoutput and is not saved anywhere.**Option B — upload a zip.** On your machine:`git archive --format=zip --output=repo.zip minh-khanh` inside the projectfolder, then run the Option B cell and upload `repo.zip`.

In [ ]:
# --- Option A: clone with a personal access token ---------------------------import os, shutil, subprocessfrom getpass import getpassOWNER, REPO, BRANCH = "LTH3ar", "seminar-2-project-01", "minh-khanh"CLEAN_URL = f"https://github.com/{OWNER}/{REPO}.git"DEST = "/content/project"token = getpass(f"GitHub token with read access to {OWNER}/{REPO}: ").strip()# subprocess rather than a ! shell magic: the URL carries the token, and IPython# echoes the command of a failing magic straight into the notebook output.shutil.rmtree(DEST, ignore_errors=True)done = subprocess.run(    ["git", "clone", "--branch", BRANCH, "--depth", "1",     f"https://{token}@github.com/{OWNER}/{REPO}.git", DEST],    capture_output=True, text=True,)if done.returncode != 0:    # Scrub the token out of any message before it is displayed.    raise SystemExit(done.stderr.replace(token, "<token>").strip())# git stores the URL it cloned from, token included. Replace it so the secret# does not sit in .git/config for the rest of the session.subprocess.run(["git", "-C", DEST, "remote", "set-url", "origin", CLEAN_URL],               check=True)del tokenos.chdir(DEST)print("remote:", subprocess.run(["git", "remote", "-v"], capture_output=True,                                text=True).stdout.splitlines()[0])print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True,                     text=True).stdout.strip())

In [ ]:
# --- Option B: upload a zip instead (skip if Option A worked) ---------------# import os, zipfile# from google.colab import files## uploaded = files.upload()                      # choose repo.zip# name = next(iter(uploaded))# os.makedirs("/content/project", exist_ok=True)# with zipfile.ZipFile(name) as archive:#     archive.extractall("/content/project")# os.chdir("/content/project")# print(sorted(os.listdir()))

## 3. Install dependencies`setfit` is installed on its own rather than through the project's `[dl]` extra:that extra pins `torch>=2.0`, and letting pip resolve it would reinstall theCUDA build Colab already has. The project package goes in with `--no-deps` forthe same reason.

In [ ]:
!pip install -q -U setfit!pip install -q -e . --no-depsimport importlib, setfit, sentence_transformers, datasets, torchprint("setfit               ", setfit.__version__)print("sentence-transformers", sentence_transformers.__version__)print("datasets             ", datasets.__version__)print("torch                ", torch.__version__, "| cuda", torch.cuda.is_available())

## 4. Smoke-test the environmentRuns the evaluation test suite, which takes a few seconds and pins the project'shand-written metrics against scikit-learn's. If the metric were wrong here, thecomparison against the published baseline further down would be meaningless, soit is worth the ten seconds.

In [ ]:
!python -m pytest tests/test_evaluation.py -q --no-header 2>&1 | tail -5

## 5. Run the reproductionTrains five models, one per project. Progress bars appear per project.This is the long cell: roughly 15 minutes on a T4. Predictions are cached to`results/setfit_predictions_official.json` as soon as training finishes, so ifthe notebook is interrupted afterwards the scoring can be redone with`--reuse-cache` instead of retraining.

In [ ]:
import osos.environ["TOKENIZERS_PARALLELISM"] = "false"os.environ["WANDB_DISABLED"] = "true"       # the organisers' notebook logs to W&Bos.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"!python experiments/03_setfit.py --preset official

## 6. Regenerate the report tables`04_make_tables.py` rewrites every table in `report/` from `results/*.json`. TheSetFit table stops being a red placeholder once the run above has produced itsresult file.

In [ ]:
!python experiments/04_make_tables.pyprint()!cat report/tables/setfit_reproduction.tex

In [ ]:
import json, pathlibdata = json.loads(pathlib.Path("results/setfit_official.json").read_text())print(f"cross-repository F1 : {data['cross_repo_f1']:.4f}")print(f"published           : {data['published']['cross-repo']:.4f}")print(f"gap                 : {data['gap']:+.4f}")print(f"95% CI              : [{data['ci'][0]:.4f}, {data['ci'][1]:.4f}]")print(f"training time       : {data['minutes']:.1f} minutes")print(f"\nVerdict: {data['verdict']}")print("\nReminder: this benchmark cannot resolve differences below 3.0 F1")print("points, so a gap inside that band is agreement, not disagreement.")

## 7. Bring the results homeDownloads the two result files. Put them in `results/` in your local checkout,re-run `python experiments/04_make_tables.py`, and commit — that is what turnsthe placeholder in the report into a real table.The prediction cache is included so the scoring can be redone offline, and sothe McNemar comparisons against the other models can be run on your own machinewithout a GPU.

In [ ]:
from google.colab import filesfor name in ("results/setfit_official.json",             "results/setfit_predictions_official.json"):    files.download(name)

---## What to do next1. Copy the two downloaded files into `results/` locally.2. `python experiments/04_make_tables.py` — the SetFit table fills in.3. Commit them on `minh-khanh`.**If the reproduction failed** (gap larger than 3 points): do not carry on tothe transformer work. Compare against the organisers' notebook cell by cell —tokenizer, `num_iterations`, `batch_size`, `num_epochs`, seed, and whether thetext is `title + " " + body`. A baseline that cannot be reproduced makes everylater comparison meaningless.**While you have a GPU:** fine-tuning RoBERTa is the obvious next use of thisruntime, and the organisers publish a template for it (`3-Template-RoBERTa.ipynb`on the `datdq` branch, scoring 0.7923). That work belongs to track C and is notwired into `ai4se.classifiers` yet.